In [1]:
# 1. Import Libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import KFold, cross_val_score, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# 2. Load Dataset
df = pd.read_csv("Clean_Titanic_Data.csv")
df.head()

# Drop the unwanted index column
if "Unnamed: 0" in df.columns:
    df = df.drop(columns=["Unnamed: 0"])

# 3. Define Features and Target
target = 'Pclass'  
X = df.drop(columns=[target])
y = df[target]

# 4. Model Training with K-Fold Cross-Validation (Before Tuning)
rf = RandomForestClassifier(random_state=42)
kf = KFold(n_splits=5, shuffle=True, random_state=42)

cv_scores = cross_val_score(rf, X, y, cv=kf, scoring='accuracy')
print("Pre-tuning Cross-Validation Accuracy Scores:", cv_scores)
print("Pre-tuning Mean Accuracy:", cv_scores.mean())

# 5. Hyperparameter Tuning using GridSearchCV
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [None, 5, 10, 15],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'bootstrap': [True, False]
}

grid_search = GridSearchCV(
    estimator=rf,
    param_grid=param_grid,
    cv=kf,
    scoring='accuracy',
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X, y)
best_rf = grid_search.best_estimator_

print("Best Hyperparameters:", grid_search.best_params_)

# 6. Model Evaluation Post-Tuning
tuned_cv_scores = cross_val_score(best_rf, X, y, cv=kf, scoring='accuracy')
print("Post-tuning Cross-Validation Accuracy Scores:", tuned_cv_scores)
print("Post-tuning Mean Accuracy:", tuned_cv_scores.mean())

# 7. Save the tuned model for deployment
import joblib

# Save the best model found by GridSearchCV
joblib.dump(best_rf, "tuned_random_forest.pkl")


Pre-tuning Cross-Validation Accuracy Scores: [0.7877095  0.70224719 0.75280899 0.74719101 0.7752809 ]
Pre-tuning Mean Accuracy: 0.7530475174188689
Fitting 5 folds for each of 216 candidates, totalling 1080 fits
Best Hyperparameters: {'bootstrap': False, 'max_depth': 10, 'min_samples_leaf': 2, 'min_samples_split': 2, 'n_estimators': 100}
Post-tuning Cross-Validation Accuracy Scores: [0.7877095  0.73033708 0.75280899 0.75842697 0.79775281]
Post-tuning Mean Accuracy: 0.7654070679806666


['tuned_random_forest.pkl']

### Reflection

#### Pre-tuning Performance
The **Random Forest classifier** achieved cross-validation accuracy scores of **[0.788, 0.702, 0.753, 0.747, 0.775]** across the 5 folds, with a **mean accuracy of approximately 0.753**.  
This indicates that the model was performing reasonably well before tuning, though there was some variation across folds.

#### Hyperparameter Tuning
Using **GridSearchCV**, **216 hyperparameter combinations** were tested.  
The best parameters found were:

- **n_estimators**: 100  
- **max_depth**: 10  
- **min_samples_split**: 2  
- **min_samples_leaf**: 2  
- **bootstrap**: False

#### Post-tuning Performance
After tuning, cross-validation accuracy scores improved to **[0.788, 0.730, 0.753, 0.758, 0.798]** with a **mean accuracy of approximately 0.765**.  
Hyperparameter tuning **clearly improved predictive performance** and slightly reduced variation between folds, resulting in a more consistent model.

#### Conclusion
K-Fold cross-validation confirmed the model's **reliability**, and **GridSearchCV** successfully optimized the Random Forest parameters.  

#### Challenges Faced
- Aligning input features in the Streamlit app to match the columns used during model training was tricky.  
- Handling the unexpected `"Unnamed: 0"` column required careful cleanup in the dataset.  
- Hyperparameter tuning with **GridSearchCV** took a long time because of the many combinations (216 total), which required patience.  
- Ensuring the saved model (`tuned_random_forest.pkl`) could be correctly loaded for real-time predictions in Streamlit required testing and debugging.

#### Key Insights from Tuning
- K-Fold Cross-Validation is crucial for understanding the **reliability** of your model before deploying it.  
- Saving and loading the trained model with `joblib` makes deployment easier and avoids retraining every time.  
- Proper preprocessing and feature alignment are just as important as tuning parameters — a mismatched column can break predictions in deployment.